In [0]:
-- =========================================
-- 1. Explode ticket custom fields
--    - One row per ticket per custom field
--    - Clean field values (remove brackets, quotes, spaces)
-- =========================================
with exploded_fields as (
 select
t.id as ticket_id
   ,field.id as field_id
   ,regexp_replace(
       regexp_replace(field.value, '["\\[\\]]', ''),
       '\\s+', ''
    ) as field_value_clean
   ,field.value as field_value_raw
 from `prod-raw-managed`.`zendesk-prod-managed-v1`.tickets t
 lateral view explode(t.fields) f as field
),
-- =========================================
-- 2. Load metadata for ticket fields
--    - Field titles
--    - Option values + labels
-- =========================================
field_metadata as (
 select
tf.id as field_id
   ,tf.raw_title
   ,opt.value as option_value
   ,opt.name as option_name
 from `prod-raw-managed`.`zendesk-prod-managed-v1`.ticket_fields tf
 lateral view outer explode(tf.custom_field_options) o as opt
),
-- =========================================
-- 3. Enrich ticket fields
--    - Resolve option labels where applicable
--    - Keep raw value when no option match
-- =========================================
enriched_fields as (
 select distinct
   ef.ticket_id
   ,coalesce(fm.raw_title, fm1.raw_title) as field_name
   ,coalesce(fm.option_name, ef.field_value_clean) as field_value
 from exploded_fields ef
 left join field_metadata fm1
   on ef.field_id = fm1.field_id
 left join field_metadata fm
   on ef.field_id = fm.field_id
  and ef.field_value_clean = fm.option_value
),
-- =========================================
-- 4. Pivot custom fields into columns
--    - One row per ticket
-- =========================================
pivoted_fields as (
 select *
 from (
   select ticket_id, field_name, field_value
   from enriched_fields
 ) src
 pivot (
   first(field_value) for field_name in (
     'Problem category',
     'Client Concerned',
     'Typology',
     'Profile (New)',
     'Profile',
     'Root Cause',
     'Offer',
     'Region concerned',
     'Category - BAS',
     'Category - KAS'
   )
 )
),
-- =========================================
-- 5. Map "Client Concerned" to Zendesk organizations
-- =========================================
client_org_mapping as (
 select distinct
   opt.name as client_name
   ,opt.value as client_value
   ,o.id as organization_id
 from (
   select
     opt.name
     ,opt.value
   from `prod-raw-managed`.`zendesk-prod-managed-v1`.ticket_fields tf
   lateral view explode(tf.custom_field_options) opts as opt
   where tf.id = 360021025018
 ) opt
 inner join `prod-raw-managed`.`zendesk-prod-managed-v1`.organizations o
   on opt.value = o.organization_fields.value_key
),
-- =========================================
-- 6. Platform companies (MySQL)
--    - Extract email domain
--    - Count users per company + domain
-- =========================================
platform_company as (
 select distinct
c.id
     ,c.name
     ,regexp_extract(em.email, '@(.+)$', 1) as domain
     ,count(u.id) as count_user
 from prod_raw.lbc_mysql.companies as c
 left join prod_raw.lbc_mysql.users as u
   on c.id = u.company_id
 inner join prod_raw.lbc_mysql.emails_prefs as em
   on u.id = em.user_id
 where c.type = 0
 group by c.id, c.name, regexp_extract(em.email, '@(.+)$', 1)
),
-- =========================================
-- 7. Keep dominant domain per company
-- =========================================
platform_company_1 as (
 select
   *
   ,row_number() over (partition by id order by count_user desc) as rn
 from platform_company
 where domain not in ('***','***.com')
),
-- =========================================
-- 8. Zendesk organizations exploded by domain
-- =========================================
zendesk_company as (
 select
   id
   ,name
   ,trim(domain_value) as domain
 from (
   select
     id
     ,name
     ,explode(domain_names) as domain_value
   from `prod-raw-managed`.`zendesk-prod-managed-v1`.organizations
 )
 where domain_value is not null
   and domain_value <> ''
),
zendesk_company_1 as (
 select *
 from zendesk_company
 where domain not in ('lbc.com','littlebigconnection.com')
),
-- =========================================
-- 9. Match Zendesk orgs to platform companies via domain
-- =========================================
company_joined as (
 select
c.id as platform_company_id
     ,z.id as zendesk_company_id
     ,c.name as company_name
     ,z.name as zendesk_name
     ,c.domain as platform_domain
     ,z.domain as zendesk_domain
     ,row_number() over (partition by z.id order by c.id) as rn
 from zendesk_company_1 z
 left join platform_company_1 c
   on z.domain = c.domain
  and c.rn = 1
 where c.id is not null
),
company_joined_1 as (
 select
   platform_company_id
   ,zendesk_company_id
   ,company_name
   ,zendesk_name
   ,platform_domain
   ,zendesk_domain
 from company_joined
 where rn = 1
),
-- =========================================
-- 10. Zendesk users with normalized roles
-- =========================================
zendesk_user as (
 select
   *
   ,case
       when email like '%@*' then 'agent'
       when email like '@*' then 'agent'
       else role
    end as user_role
   ,row_number() over (
       partition by u.id, u.email
       order by updated_at asc
    ) as row_number
 from `prod-raw-managed`.`zendesk-prod-managed-v1`.users as u
),
-- =========================================
-- 11. Ticket audits with author role
-- =========================================
ticket_audits_with_user as (
 select
   ta.ticket_id
   ,ta.audit_id
   ,ta.author_id
   ,coalesce(u.user_role, 'end-user') as role
   ,cast(ta.created_at as timestamp) as created_at
   ,ta.event_id
   ,ta.event_type
   ,ta.public
   ,ta.body
   ,cast(tm.solved_at as timestamp) as solved_at
 from `prod-raw-managed`.`zendesk-prod-managed-v1`.ticket_audits as ta
 left join `prod-raw-managed`.`zendesk-prod-managed-v1`.ticket_metrics tm
   on tm.ticket_id = ta.ticket_id
 left join zendesk_user u
   on u.id = ta.author_id
  and row_number = 1
 where event_type = 'Comment'
   and ta.public = true
),
-- =========================================
-- 12. Compute next agent response timestamp
-- =========================================
with_agent_response as (
 select
   *
   ,min(case when role = 'agent' then created_at end)
     over (
       partition by ticket_id
       order by audit_id
       rows between 1 following and unbounded following
     ) as next_agent_created_at
 from ticket_audits_with_user
),
-- =========================================
-- 13. SLA computation
--    - Total hours
--    - Weekend removal
--    - Business hours
--    - SLA status (48 business hours)
-- =========================================
zendesk_audit as (
 select
   ticket_id
   ,audit_id
   ,author_id
   ,role
   ,lead(role) over (partition by ticket_id order by audit_id) as next_role
   ,created_at
   ,next_agent_created_at
   ,solved_at
   ,coalesce(next_agent_created_at, solved_at, current_timestamp()) as reference_timestamp
   -- Time metrics
   ,datediff(reference_timestamp, created_at) as total_days
   ,(unix_timestamp(reference_timestamp) - unix_timestamp(created_at)) / 3600.0 as total_hours
   -- Weekend days
   ,(
     floor(datediff(reference_timestamp, created_at) / 7) * 2
     +
     case
       when dayofweek(created_at) in (1,7) then 1
       else 0
     end
   ) as weekend_days
   -- Business hours
   ,total_hours - (weekend_days * 24) as business_hours
   -- SLA logic (48 business hours)
   ,case
     when role = 'end-user' and business_hours <= 48 then 'In SLA'
     when role = 'end-user' and business_hours > 48 then 'Out SLA'
     else null
   end as sla_status
   ,event_id
   ,event_type
   ,public
   ,body
 from with_agent_response
),
-- =========================================
-- 14. Final Zendesk fact table
-- =========================================
zendesk_fact as (
select distinct
t.id as ticket_id
 ,concat('https://littlebigconnection.zendesk.com/agent/tickets/', t.id) as ticket_url
 ,coalesce(cj.platform_company_id, 2) as platform_company_id
 ,coalesce(t.organization_id, com.organization_id) as organization_id
 ,p.`Client Concerned` as client_concerned
 ,coalesce(p.`Profile (New)`, p.`Profile`) as requester_type
 ,p.`Typology` as typology
 ,p.`Problem category` as problem_category
 ,za.sla_status
 ,za.business_hours
 ,cast(t.created_at as timestamp) as created_at
from `prod-raw-managed`.`zendesk-prod-managed-v1`.tickets t
left join pivoted_fields p on t.id = p.ticket_id
left join client_org_mapping com on p.`Client Concerned` = com.client_name
left join company_joined_1 cj on coalesce(t.organization_id, com.organization_id) = cj.zendesk_company_id
left join zendesk_audit za on t.id = za.ticket_id
where p.`Typology` in ('Admin & Finance', 'BAS', 'KAP', 'KAS')
)
-- =========================================
-- 15. Monthly SLA aggregation
-- =========================================
select
 month(created_at) as month
 ,count(distinct ticket_id) as total_tickets
 ,count(distinct audit_id) as total_audits
 ,count(case when sla_status = 'In SLA' then audit_id end) as audits_in_sla
 ,count(case when sla_status = 'Out SLA' then audit_id end) as audits_out_sla
 ,count(case when sla_status is null then audit_id end) as audits_wo_sla
 ,round(avg(case when sla_status is not null then business_hours end), 2)
   as avg_business_hours_to_response
from zendesk_fact
where year(created_at) = 2025
 and organization_id = *
group by month(created_at)
order by month(created_at);